# 阶段 1.5（进阶）：MoE Transformer——稀疏激活与专家混合

## 学习目标
- [ ] 理解 MoE 的核心价值：**参数量与计算量分离**（更多容量，不等于更多计算）
- [ ] 能手写 `Router`（Top-K 门控）和 `MoELayer`（多 Expert 加权聚合）
- [ ] 理解 **Expert Collapse** 问题，以及辅助负载均衡损失如何解决它
- [ ] 对比 Dense 和 MoE 的训练曲线，解释差异原因
- [ ] 能回答：「Mixtral-8x7B 实际计算量相当于几个 Dense 7B？」

## 前置知识（必须）
- 完成阶段 1（`01_transformer_from_scratch.ipynb`）
- 理解 FeedForward 层的结构（d → 4d → d）
- 了解 `nn.ModuleList` vs `nn.Sequential` 的区别

## 预计时间
2-3 小时（含训练时间）| **硬件**：CPU 即可（d_model=64 教学规模）

## 配套资源
- 完整实现：`src/moe_model.py`
- 教学笔记：`experiments/exp_005_moe_transformer.md`
- Wiki 概念：`wiki/concepts/mixture_of_experts.md`

---

## Part 1：为什么需要 MoE？

### Dense Transformer 的扩展问题

在阶段 1 中，每个 token 都经过**完全相同**的 FeedForward 网络（FFN）：

```
token 1 → [FFN 权重 W] → 输出
token 2 → [FFN 权重 W] → 输出  （同一套 W！）
token 3 → [FFN 权重 W] → 输出
```

**问题**：增大模型能力 = 增大 FFN → 参数量 ↑，计算量 ↑，两者**同比例**增加。

### MoE 的思路：条件计算

> 「不同 token 需要的知识不同——让 Router 动态分配每个 token 去哪个 Expert」

```
                     ┌── Expert 0（数学？）──┐
token → [Router] ────┤                       ├──→ 加权求和 → 输出
                     └── Expert 3（语言？）──┘
              （Expert 1, 2 对此 token 沉默）
```

**关键收益**（以 4 experts, top_k=2 为例）：
| 维度 | Dense FFN | MoE（4E2K） |
|------|-----------|-------------|
| 参数量 | 1× | **4×**（更多容量） |
| 每 token 计算量 | 1× | **2×**（只有 top_k 个激活） |
| 效率增益 | — | 花 2 倍计算，获 4 倍容量 |

In [ ]:
# Cell 3：对比 Dense vs MoE 的参数量
# 运行前确保你在 ai-practice/ 目录
import sys, os
sys.path.insert(0, 'src')  # 让 Python 能找到 src/ 下的模块

# 注意：直接 import model 会触发训练循环（model.py 顶层有训练代码）
# 所以我们用手动参数计算来对比
import torch

# ---- 超参数（与两个模型相同）----
d_model = 64
num_blocks = 8
num_heads = 4
num_experts = 4  # MoE 专有
top_k = 2        # MoE 专有

# ---- 计算各组件的参数量 ----
# FFN 参数量：Linear(d→4d) + Linear(4d→d) = d×4d + 4d×d = 8d²
ffn_params = d_model * (d_model * 4) + (d_model * 4) * d_model

# Attention 参数量：Q/K/V 各 d×head_size，输出投影 d×d
# head_size = d_model / num_heads，多头合并后总参数 = 4 × d × d
attn_params = 4 * d_model * d_model  # Q+K+V+proj

# LayerNorm 参数量：2 × d（per block，2 个 LN）
ln_params = 2 * 2 * d_model

# Dense 每个 Block 的参数量
dense_block_params = attn_params + ffn_params + ln_params

# MoE 每个 Block 的参数量（FFN 变成 num_experts 倍，加上 Router）
router_params = d_model * num_experts  # Linear(d_model → num_experts)
moe_block_params = attn_params + num_experts * ffn_params + router_params + ln_params

print("=" * 55)
print(f"  Dense Transformer vs MoE Transformer 参数对比")
print("=" * 55)
print(f"  d_model={d_model}, num_blocks={num_blocks}")
print(f"  MoE: num_experts={num_experts}, top_k={top_k}")
print("-" * 55)
print(f"  每个 Block 的参数量:")
print(f"    Attention（共享）: {attn_params:>10,}")
print(f"    Dense FFN:         {ffn_params:>10,}")
print(f"    MoE FFN（4 exp）:  {num_experts * ffn_params:>10,}  （{num_experts}×）")
print(f"    Router（MoE 专有）:{router_params:>10,}")
print("-" * 55)
print(f"  所有 {num_blocks} Blocks 合计:")
print(f"    Dense:  {num_blocks * dense_block_params:>12,}")
print(f"    MoE:    {num_blocks * moe_block_params:>12,}  （{num_blocks * moe_block_params / (num_blocks * dense_block_params):.1f}×）")
print("-" * 55)
print(f"  每 token 激活的 Expert 数: {top_k}/{num_experts} = {top_k/num_experts:.0%}")
print(f"  每 token 的计算量（FFN部分）: Dense 的 {top_k}×")
print("=" * 55)

---

## Part 2：Router——Token 路由器

Router 的任务：给定 token 表示 $x \in \mathbb{R}^{d_{model}}$，决定发送给哪些 Expert。

### 完整流程（top_k=2, num_experts=4）

```
x ∈ R^64
  ↓  Linear(64 → 4)，无 bias
logits = [1.2,  -0.3,  0.8,  2.1]
  ↓  Softmax
gate_probs = [0.25, 0.05, 0.18, 0.52]   ← 原始概率（用于 aux_loss）
  ↓  Top-2 选择
选中 Expert 3（0.52）和 Expert 0（0.25）
  ↓  重新归一化（sum=1）
weights = [0.33, 0.67]（Expert 0: 0.33，Expert 3: 0.67）
  ↓
输出 = 0.33 × Expert_0(x) + 0.67 × Expert_3(x)
```

**为什么要重新归一化？** 确保输出的尺度不受绝对门控值影响，梯度更稳定。

In [ ]:
# Cell 5：Router 实现与 token 分配可视化
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'sans-serif'  # 避免中文字体问题

d_model = 64
num_experts = 4
top_k = 2
torch.manual_seed(42)

class Router(nn.Module):
    """Token 路由器：决定每个 token 发送给哪些 Expert"""
    def __init__(self):
        super().__init__()
        # 线性门控：d_model → num_experts（无 bias，纯粹的内容路由）
        self.gate = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x):
        # x: (N, d_model)，N = B × T
        gate_logits = self.gate(x)                        # (N, num_experts)
        gate_probs = F.softmax(gate_logits, dim=-1)       # (N, num_experts)

        # Top-K 选择
        top_k_weights, top_k_indices = torch.topk(gate_probs, top_k, dim=-1)

        # 重新归一化（使选中 Expert 的权重 sum=1）
        top_k_weights = top_k_weights / top_k_weights.sum(dim=-1, keepdim=True)

        return top_k_indices, top_k_weights, gate_probs

# 模拟 1 个 batch × 16 个 token 的输入
router = Router()
dummy_input = torch.randn(16, d_model)  # 16 个 token

indices, weights, gate_probs = router(dummy_input)

print(f"输入形状: {dummy_input.shape}")
print(f"选中的 Expert 编号 (top_k={top_k}): {indices.shape}")
print(f"  前 4 个 token 的分配: {indices[:4].tolist()}")
print(f"  对应门控权重:         {weights[:4].round(decimals=3).tolist()}")
print(f"  权重之和（应为 1.0）: {weights[:4].sum(dim=-1).tolist()}")

# 统计每个 Expert 的 top-1 接收比例
top1 = indices[:, 0]  # top-1 分配
print(f"\n各 Expert 的 top-1 接收比例（随机初始化，应接近均匀）:")
for e in range(num_experts):
    count = (top1 == e).sum().item()
    print(f"  Expert {e}: {count:2d} tokens（{count/len(top1):.0%}）")

# 可视化门控概率热图
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 热图：每个 token 的门控概率分布
axes[0].imshow(gate_probs.detach().numpy(), aspect='auto', cmap='Blues')
axes[0].set_xlabel('Expert Index')
axes[0].set_ylabel('Token Index')
axes[0].set_title('Gate Probabilities (random init)')
axes[0].set_xticks(range(num_experts))
axes[0].set_xticklabels([f'E{i}' for i in range(num_experts)])
plt.colorbar(axes[0].images[0], ax=axes[0])

# 柱状图：Expert 接收 token 比例
expert_load = [(top1 == e).sum().item() for e in range(num_experts)]
axes[1].bar([f'Expert {i}' for i in range(num_experts)], expert_load,
            color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'])
axes[1].axhline(y=len(top1) / num_experts, color='red', linestyle='--',
                label=f'均匀分配目标 ({len(top1)/num_experts:.0f} tokens)')
axes[1].set_ylabel('Number of tokens (top-1 assignment)')
axes[1].set_title('Expert Load Distribution (random init)')
axes[1].legend()

plt.tight_layout()
plt.show()
print("\n💡 随机初始化时，Router 接近均匀分配 token。训练后会出现分化——这是 Expert 专业化的表现。")

---

## Part 3：MoELayer——多 Expert 加权聚合

MoELayer 的工作流程：

```
x: (B, T, d_model)
  ↓ reshape → (B*T, d_model)
  ↓ Router(x)
  → indices (B*T, top_k)：每个 token 选中的 Expert 编号
  → weights (B*T, top_k)：对应的门控权重
  ↓ 逐 Expert 处理（稀疏计算）
  for expert_i in experts:
      找出被分配到 expert_i 的 token
      output += weight × expert_i(token)
  ↓ reshape → (B, T, d_model)
  + 计算 aux_loss（负载均衡）
```

In [ ]:
# Cell 7：完整 MoELayer 实现
import torch
import torch.nn as nn
import torch.nn.functional as F

d_model = 64
num_experts = 4
top_k = 2
dropout_rate = 0.1

class FeedForward(nn.Module):
    """单个 Expert（与 Dense Transformer 中的 FFN 完全相同）"""
    def __init__(self):
        super().__init__()
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout_rate),
        )
    def forward(self, x):
        return self.ffn(x)

class Router(nn.Module):
    def __init__(self):
        super().__init__()
        self.gate = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x):
        gate_probs = F.softmax(self.gate(x), dim=-1)
        top_k_weights, top_k_indices = torch.topk(gate_probs, top_k, dim=-1)
        top_k_weights = top_k_weights / top_k_weights.sum(dim=-1, keepdim=True)
        return top_k_indices, top_k_weights, gate_probs

class MoELayer(nn.Module):
    """核心 MoE 层：Router + N 个 Expert + 辅助损失"""
    def __init__(self):
        super().__init__()
        self.router = Router()
        # ModuleList：PyTorch 能正确追踪所有 Expert 的参数
        self.experts = nn.ModuleList([FeedForward() for _ in range(num_experts)])

    def forward(self, x):
        B, T, C = x.shape
        x_flat = x.view(B * T, C)   # 合并 batch 和 seq 维度，统一处理

        # ① 路由
        indices, weights, gate_probs = self.router(x_flat)

        # ② Expert 计算（稀疏：每个 Expert 只处理被分配到它的 token）
        output = torch.zeros_like(x_flat)
        for expert_idx, expert in enumerate(self.experts):
            # 找出分配到此 Expert 的 token（可能是 top-1 或 top-2）
            token_mask, k_pos = (indices == expert_idx).nonzero(as_tuple=True)
            if token_mask.numel() == 0:
                continue  # 此 Expert 本 batch 内无 token，跳过（节省计算）

            expert_output = expert(x_flat[token_mask])          # 只处理分配到的 token
            gate_weight = weights[token_mask, k_pos].unsqueeze(-1)  # 对应门控权重
            output[token_mask] += gate_weight * expert_output   # 加权累加

        # ③ 辅助负载均衡损失（Switch Transformer 风格）
        with torch.no_grad():
            top1_indices = indices[:, 0]    # 取 top-1 分配（用于统计 f_i）
            fraction = torch.zeros(num_experts)
            for e in range(num_experts):
                fraction[e] = (top1_indices == e).float().mean()
        mean_gate = gate_probs.mean(dim=0)  # P_i，可微分
        aux_loss = num_experts * (fraction * mean_gate).sum()

        return output.view(B, T, C), aux_loss

# 验证 MoELayer
torch.manual_seed(42)
moe_layer = MoELayer()
dummy_x = torch.randn(2, 4, d_model)  # batch=2, seq=4, d_model=64

output, aux_loss = moe_layer(dummy_x)
print(f"输入形状:  {dummy_x.shape}")
print(f"输出形状:  {output.shape}   ← 与输入相同")
print(f"aux_loss:  {aux_loss.item():.6f}")
print(f"\n理想的 aux_loss ≈ 1.0（均匀分配时 f_i=1/N，P_i=1/N，sum=N×(1/N×1/N)×N=1.0）")
print(f"当前值接近 1.0，说明随机初始化时 Router 接近均匀分配。")

---

## Part 4：Expert Collapse 与辅助损失

### 没有 aux_loss 会发生什么？

**Expert Collapse 的正反馈循环**：
```
Expert 0 偶然被选中更多
  → Expert 0 获得更多梯度
  → Expert 0 变得更强（损失更低）
  → Router 更倾向于选 Expert 0
  → Expert 0 被选中更多
  → ... （正反馈，最终只有 Expert 0 被使用）
```

### aux_loss 的作用

$$\mathcal{L}_{aux} = \alpha \cdot N \cdot \sum_{i=1}^{N} f_i \cdot P_i$$

- $f_i$：Expert $i$ 接收的 token 比例（不可微，用 `stop_gradient`）
- $P_i$：Expert $i$ 的平均门控概率（可微，用于梯度传播）
- 不均衡 → 某个 $f_i$ 很大 → 对应的 $f_i \times P_i$ 大 → 损失大 → 推动 Router 均匀化

In [ ]:
# Cell 9：演示 Expert Collapse（无 aux_loss vs 有 aux_loss）
# 注意：这是一个简化演示，使用随机梯度来模拟训练偏差
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
d_model = 64
num_experts = 4
top_k = 2

def simulate_routing(use_aux_loss, steps=200):
    """模拟 Router 的训练过程，追踪 Expert 负载变化"""
    gate = nn.Linear(d_model, num_experts, bias=False)
    optimizer = torch.optim.SGD(gate.parameters(), lr=0.1)

    # 模拟「Expert 0 更好」：向 Expert 0 倾斜的假损失
    # （真实训练中，Expert 0 因为多被选中而训练得更好，产生相同效果）
    target_bias = torch.tensor([2.0, 0.0, 0.0, 0.0])  # 偏向 Expert 0

    load_history = []

    for step in range(steps):
        x = torch.randn(64, d_model)  # 64 个 token
        gate_probs = F.softmax(gate(x), dim=-1)
        top_k_weights, top_k_indices = torch.topk(gate_probs, top_k, dim=-1)

        # 统计负载
        top1 = top_k_indices[:, 0]
        load = torch.tensor([(top1 == e).float().mean().item() for e in range(num_experts)])
        load_history.append(load.tolist())

        # 主损失：模拟向 Expert 0 倾斜的梯度
        main_loss = -((gate_probs * target_bias).sum())

        if use_aux_loss:
            # 辅助损失：鼓励均匀分配
            with torch.no_grad():
                fraction = torch.zeros(num_experts)
                for e in range(num_experts):
                    fraction[e] = (top1 == e).float().mean()
            mean_gate = gate_probs.mean(dim=0)
            aux_loss = num_experts * (fraction * mean_gate).sum()
            loss = main_loss + 0.5 * aux_loss  # α=0.5（演示用，比实际大）
        else:
            loss = main_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return load_history

# 模拟两种情况
history_no_aux = simulate_routing(use_aux_loss=False)
history_with_aux = simulate_routing(use_aux_loss=True)

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
steps = list(range(len(history_no_aux)))

for i, (ax, history, title) in enumerate([
    (axes[0], history_no_aux, 'Without aux_loss (Expert Collapse)'),
    (axes[1], history_with_aux, 'With aux_loss (Load Balanced)'),
]):
    for e in range(num_experts):
        load_e = [h[e] for h in history]
        ax.plot(steps, load_e, label=f'Expert {e}', color=colors[e], linewidth=2)
    ax.axhline(y=0.25, color='gray', linestyle='--', alpha=0.7, label='Ideal (25%)')
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Token Fraction (top-1 assignment)')
    ax.set_title(title)
    ax.legend(loc='upper right')
    ax.set_ylim([0, 1])
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print("\n左图：无 aux_loss → Expert 0 逐渐垄断所有 token（Expert Collapse）")
print("右图：有 aux_loss → 各 Expert 的负载维持在均匀水平附近")

In [ ]:
# Cell 10：训练 Dense Transformer（基准对比）
# ⏱ CPU 约需 30-60 分钟（5000 步），GPU 约 5 分钟
# 如果只想看效果，可以减少 max_iters（如改为 500 步）
import subprocess
import sys

print("Dense Transformer 训练说明：")
print("  代码位于 src/model.py")
print("  命令行运行：python3 src/model.py（从 ai-practice/ 目录）")
print("  完成后会生成 model-ckpt.pt")
print()
print("若已有 model-ckpt.pt，可直接跳到 Cell 11 训练 MoE 模型。")
print()

# 在 notebook 中直接运行（取消下方注释）
# 注意：这会在当前进程中执行，输出会打印在此 cell 下方
# import importlib.util, sys, os
# spec = importlib.util.spec_from_file_location("model", "src/model.py")
# mod = importlib.util.load_from_spec(spec)  # 会触发训练

# 推荐方式：在终端中运行
print("推荐：打开终端，cd 到 ai-practice/，然后运行：")
print("  python3 src/model.py")

In [ ]:
# Cell 11：训练 MoE Transformer
# ⏱ CPU 约需 45-90 分钟（5000 步，每步计算量约 Dense 的 2×）
# 完成后会生成 moe-model-ckpt.pt
print("MoE Transformer 训练说明：")
print("  代码位于 src/moe_model.py")
print("  命令行运行：python3 src/moe_model.py（从 ai-practice/ 目录）")
print("  完成后会生成 moe-model-ckpt.pt")
print()
print("训练过程中你会看到：")
print("  Step:     0 | Train: 11.xxxx | Valid: 11.xxxx  ← 初始损失≈ln(vocab_size)")
print("  Step:    50 | Train: 8.xxxx  | Valid: 8.xxxx")
print("  ...")
print("  Step:  5000 | Train: X.xxxx  | Valid: X.xxxx")
print()
print("训练完成后，回到这个 notebook 继续 Cell 12-13 的可视化。")

In [ ]:
# Cell 12：训练曲线对比图
# 运行此 Cell 前，请先完成 Cell 10-11 的训练，并将损失数据填入下方
import matplotlib.pyplot as plt

# ====== 请将实际训练数据填入此处 ======
# 格式：(step, train_loss, val_loss)
# 从训练日志中复制，或按 eval_interval=50 记录

dense_log = [
    # (0, 11.xx, 11.xx),
    # (50, 8.xx, 8.xx),
    # ... 替换为实际值
]

moe_log = [
    # (0, 11.xx, 11.xx),
    # (50, 8.xx, 8.xx),
    # ... 替换为实际值
]

if not dense_log or not moe_log:
    print("⚠️  请先完成训练（Cell 10-11），然后将损失日志填入 dense_log 和 moe_log。")
    print("   格式示例：")
    print("   dense_log = [(0, 11.52, 11.51), (50, 8.23, 8.31), ...]")
else:
    dense_steps = [x[0] for x in dense_log]
    dense_train = [x[1] for x in dense_log]
    dense_val = [x[2] for x in dense_log]

    moe_steps = [x[0] for x in moe_log]
    moe_train = [x[1] for x in moe_log]
    moe_val = [x[2] for x in moe_log]

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(dense_steps, dense_train, 'b-', label='Dense Train', linewidth=2)
    ax.plot(dense_steps, dense_val, 'b--', label='Dense Val', linewidth=1.5, alpha=0.7)
    ax.plot(moe_steps, moe_train, 'r-', label='MoE Train', linewidth=2)
    ax.plot(moe_steps, moe_val, 'r--', label='MoE Val', linewidth=1.5, alpha=0.7)
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Loss')
    ax.set_title('Dense vs MoE Transformer: Training Curves')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    final_dense = dense_val[-1]
    final_moe = moe_val[-1]
    print(f"最终验证损失：Dense = {final_dense:.4f}，MoE = {final_moe:.4f}")
    if final_moe < final_dense:
        print(f"MoE 更低 {(final_dense - final_moe):.4f}（约 {(final_dense - final_moe)/final_dense:.1%}）")
    else:
        print("Dense 和 MoE 在此规模下性能接近（d_model=64 较小，差异可能不显著）")

In [ ]:
# Cell 13：Expert 利用率可视化（训练后）
import torch
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, 'src')

# 尝试加载训练好的 MoE 模型
try:
    # 动态加载 moe_model.py，避免触发训练主循环
    import importlib.util
    spec = importlib.util.spec_from_file_location("moe_module", "src/moe_model.py")
    moe_module = importlib.util.load_module_from_spec(spec)
    spec.loader.exec_module(moe_module)

    model = moe_module.MoETransformerLanguageModel()
    model.load_state_dict(torch.load('moe-model-ckpt.pt', map_location='cpu'))
    model.eval()

    # 采样一批数据，统计 Expert 利用率
    n_batches = 20
    all_expert_loads = [torch.zeros(4) for _ in range(8)]  # 8 blocks × 4 experts

    with torch.no_grad():
        for _ in range(n_batches):
            x, _ = moe_module.get_batch('valid')
            emb = model.token_embedding_lookup_table(x)  # (B, T, d_model)

            # 逐 Block 提取路由信息
            cur_x = emb
            for block_idx, block in enumerate(model.transformer_blocks):
                ln1_out = block.layer_norm_1(cur_x)
                attn_out = block.multi_head_attention_layer(ln1_out)
                cur_x = cur_x + attn_out

                ln2_out = block.layer_norm_2(cur_x)
                x_flat = ln2_out.view(-1, 64)
                router = block.moe_layer.router
                indices, weights, _ = router(x_flat)
                top1 = indices[:, 0]
                for e in range(4):
                    all_expert_loads[block_idx][e] += (top1 == e).float().mean()

                moe_out, _ = block.moe_layer(ln2_out)
                cur_x = cur_x + moe_out

    # 归一化
    expert_matrix = torch.stack(all_expert_loads) / n_batches  # (8, 4)

    # 热图可视化
    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(expert_matrix.numpy(), cmap='RdYlGn', aspect='auto', vmin=0, vmax=0.5)
    ax.set_xlabel('Expert Index')
    ax.set_ylabel('Block Index')
    ax.set_title('Expert Utilization Heatmap (trained model)\n(green=balanced, red=overloaded)')
    ax.set_xticks(range(4))
    ax.set_xticklabels([f'Expert {i}' for i in range(4)])
    ax.set_yticks(range(8))
    ax.set_yticklabels([f'Block {i}' for i in range(8)])
    plt.colorbar(im, ax=ax, label='Token fraction (top-1)')
    plt.tight_layout()
    plt.show()
    print("\n绿色 ≈ 均匀（约 25%），黄/红色 = 过载，深绿 = 闲置")

except FileNotFoundError:
    print("⚠️  未找到 moe-model-ckpt.pt")
    print("   请先运行 python3 src/moe_model.py 完成训练")
except Exception as e:
    print(f"加载失败：{e}")
    print("请确保 moe-model-ckpt.pt 是由当前版本的 moe_model.py 生成的")

---

## 本章小结

### 核心知识点

| 概念 | 关键点 |
|------|-------|
| **MoE 核心价值** | 参数量与计算量分离：`num_experts × 参数`，但只用 `top_k × 计算` |
| **Router** | Linear(d→N) + Softmax + Top-K + 重新归一化 |
| **MoELayer** | 逐 Expert 稀疏计算 + 门控加权聚合 |
| **Expert Collapse** | 正反馈导致 Router 退化，aux_loss 打破循环 |
| **aux_loss** | $N \times \sum f_i P_i$，$f_i$ stop_grad，$P_i$ 可微 |
| **与 Dense 的唯一区别** | `TransformerBlock.feed_forward` → `MoELayer`；不能用 `nn.Sequential` |

### 思考题

1. **核心**：Mixtral-8x7B 有 8 个 Expert，激活 top_k=2。FFN 占模型参数约 2/3，实际推理计算量相当于几个 Dense 7B 模型？（提示：注意 Attention 层不是 MoE）

2. **动手**：将 `top_k` 从 2 改为 1，训练 500 步。Expert 利用率分布是否更不均衡？为什么？

3. **思考**：为什么 MoELayer 中不能用 `nn.Sequential` 来串联 Transformer Block？如果非要用，如何改造？

4. **数学**：当 `num_experts=4, top_k=2, d_model=64` 时，MoELayer 的参数量是多少？与同规模 Dense FFN 相比，额外增加了多少参数（百分比）？

5. **延伸**：DeepSeek-V3 使用 `num_experts=256, top_k=8`。这种「细粒度」MoE（相比 Mixtral 的 8E2K）有什么优缺点？

### 下一步

- **→ 阶段 2**：[02_transformers_library.ipynb](02_transformers_library.ipynb)——加载真实的 MoE 模型（如 Mixtral）
- **→ 阶段 4**：[04_qwen25_grpo_finetuning.ipynb](04_qwen25_grpo_finetuning.ipynb)——在 Qwen 模型上用 LoRA + GRPO 微调
- **深入阅读**：[Switch Transformer 论文](https://arxiv.org/abs/2101.03961)（MoE 负载均衡损失的原始提出）